# Local RAG Assistant for Customer Support Ticket Triage

Runs entirely on your Mac: **MiniLM** finds similar past tickets, **Llama 3.2 (via Ollama)** writes the triage.

Run the cells top to bottom. The first run downloads the MiniLM model (~80 MB) once.
Make sure Ollama is running before the generation cells (it listens on `localhost:11434`).

## 1. Configuration

In [1]:
import os, glob, time, json
import pandas as pd

# Resolve data/ whether the notebook is launched from the project root or from
# notebooks/ (nbconvert runs with cwd = the notebook's own folder).
DATA_DIR = "data" if os.path.isdir("data") else os.path.join("..", "data")

EMBED_MODEL = "all-MiniLM-L6-v2"        # embedding model (downloads on first use)
GEN_MODEL   = "llama3.2:3b"             # local model served by Ollama
OLLAMA_URL  = "http://localhost:11434/api/generate"
K           = 3                          # how many past tickets to retrieve
KB_SIZE     = 1500                       # how many tickets go in the knowledge base

print("DATA_DIR ->", os.path.abspath(DATA_DIR))


DATA_DIR -> /Users/juampa/Documents/AtlantisUniversity/Master/MAI600/finalProject/data


## 2. Load the dataset and keep the English tickets

In [2]:
# Pinned to the project dataset (documented in the proposal). Do NOT auto-pick by
# file size: data/ holds several Kaggle variants and the largest one is a different file.
CSV_PATH = os.path.join(DATA_DIR, "dataset-tickets-multi-lang-4-20k.csv")

df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH)
print("Total rows:", len(df))
print("Languages:", df["language"].value_counts().to_dict())

# English only. The multilingual rows would pollute both the knowledge base and
# the embeddings, and the triage output has to be in English.
df = df[df["language"] == "en"].copy()
df = df.dropna(subset=["body", "answer"]).reset_index(drop=True)
print("English rows with body + answer:", len(df))
df[["subject", "queue", "type", "priority"]].head()


Loaded: ../data/dataset-tickets-multi-lang-4-20k.csv
Total rows: 20000
Languages: {'en': 11923, 'de': 8077}
English rows with body + answer: 11919


,subject,queue,type,priority
0,Customer Support Inquiry,Customer Service,Request,medium
1,Data Analytics for Investment,Customer Service,Request,medium
2,Security,Customer Service,Request,medium
3,Concerns About Securing Medical Data on 2-in-1...,Technical Support,Request,medium
4,Problem with Integration,IT Support,Problem,high


## 3. Build the knowledge base

A subset of past tickets. We search on the customer's problem (`body`) and later show the agent's `answer` as guidance.

In [3]:
kb = df.sample(n=min(KB_SIZE, len(df)), random_state=42).reset_index(drop=True)
kb_texts = (kb["subject"].fillna("") + ". " + kb["body"]).tolist()
print("Knowledge base size:", len(kb))

Knowledge base size: 1500


## 4. Embed the knowledge base with MiniLM

First run downloads the model (~80 MB), then it's cached.

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
kb_emb = embedder.encode(kb_texts, convert_to_numpy=True, show_progress_bar=True).astype("float32")
print("Embeddings shape:", kb_emb.shape)   # (KB_SIZE, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Embeddings shape: (1500, 384)


## 5. Build the FAISS index

Cosine similarity via inner product on normalized vectors.

In [5]:
import faiss

faiss.normalize_L2(kb_emb)
index = faiss.IndexFlatIP(kb_emb.shape[1])
index.add(kb_emb)
print("Indexed vectors:", index.ntotal)

Indexed vectors: 1500


## 6. Retrieval: find the K most similar past tickets

In [6]:
def retrieve(query, k=K):
    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, idx = index.search(q, k)
    hits = []
    for rank, (i, s) in enumerate(zip(idx[0], scores[0]), start=1):
        row = kb.iloc[int(i)]
        hits.append({"n": rank, "score": float(s),
                     "subject": row["subject"], "body": row["body"],
                     "answer": row["answer"], "queue": row["queue"],
                     "type": row["type"], "priority": row["priority"]})
    return hits

## 7. Build a grounded, cited prompt

In [7]:
def build_prompt(ticket_text, hits):
    context = ""
    for h in hits:
        context += f"[{h['n']}] (queue: {h['queue']}, type: {h['type']}, priority: {h['priority']})\n"
        context += f"Customer: {h['body']}\n"
        context += f"Agent answer: {h['answer']}\n\n"
    prompt = (
        "You are a support triage assistant. Using ONLY the past tickets below, "
        "triage the NEW ticket. Cite the past tickets you use with [1], [2], etc. "
        "If the past tickets do not cover it, say there is not enough information.\n\n"
        f"PAST TICKETS:\n{context}"
        f"NEW TICKET:\n{ticket_text}\n\n"
        "Answer with: Suggested queue, Type, Priority, Draft first response, and Citations."
    )
    return prompt

## 8. Call the local model through Ollama

Ollama must be running (`ollama serve` happens automatically when the app is open).

In [8]:
import requests

def ask_ollama(prompt, model=GEN_MODEL):
    t0 = time.time()
    r = requests.post(OLLAMA_URL, json={"model": model, "prompt": prompt, "stream": False})
    r.raise_for_status()
    return r.json()["response"], time.time() - t0

## 9. Try it on one incoming ticket

We take a random English ticket as the "new" one so you can compare the model's triage against the real labels.

In [9]:
sample = df.sample(1, random_state=7).iloc[0]
incoming = f"{sample['subject']}. {sample['body']}"

print("INCOMING TICKET:\n", incoming[:400], "\n")
print("TRUE labels -> queue:", sample["queue"], "| type:", sample["type"], "| priority:", sample["priority"], "\n")

hits = retrieve(incoming)
answer, secs = ask_ollama(build_prompt(incoming, hits))
print("=== MODEL TRIAGE (with RAG) ===")
print(answer)
print(f"\n(generated locally in {secs:.1f}s)")

INCOMING TICKET:
 nan. A project management SaaS application is encountering functionality problems across multiple devices. Recent updates might be causing integration conflicts. Despite clearing the cache, reinstalling the application, and ensuring all updates are installed, the issue continues. Assistance is needed to resolve this problem. 

TRUE labels -> queue: Technical Support | type: Incident | priority: high 



=== MODEL TRIAGE (with RAG) ===
Based on the past tickets provided, I would recommend the following triage for the new ticket:

Queue: Technical Support
Type: Problem
Priority: Medium [1], [2]

Draft first response:
"Hello [name], we're here to help with the functionality problems in your project management SaaS application. We've seen similar issues with recent updates causing integration conflicts, but we'd like more information on the specific devices and software involved. Could you please provide details on the devices affected and any recent API updates or server changes made? Additionally, it would be helpful to know if you've encountered any error messages. Please let us know a suitable time for a call to discuss further."

Note that I'm not recommending high priority as [3] has already been marked as high priority due to critical issues with the task sync feature, and this new ticket seems like a similar but less urgent issue.

(generated locally in 5.1s)


## 10. Baseline: same model, NO retrieval

This is the baseline for Section 9 of your proposal. Compare its triage to the RAG one above.

In [10]:
baseline_prompt = (
    "You are a support triage assistant. Triage this ticket. "
    "Give Suggested queue, Type, Priority, and a Draft first response.\n\n"
    f"NEW TICKET:\n{incoming}"
)
base_answer, base_secs = ask_ollama(baseline_prompt)
print("=== BASELINE TRIAGE (no retrieval) ===")
print(base_answer)
print(f"\n(baseline, {base_secs:.1f}s)")

=== BASELINE TRIAGE (no retrieval) ===
Based on the information provided, I'm triaging this ticket as follows:

**Suggested Queue:** Development Support (DS)

**Type:** Application Issue

**Priority:** Medium

**Draft First Response:**

"Hello,

Thank you for reaching out about the project management SaaS application issue you're experiencing. We apologize for the inconvenience this is causing and are here to help.

Can you please provide more details about the issue, such as:

- The specific functionality that's not working
- Any error messages or screens you see when trying to access the application
- The devices and browsers used to experience the issue

Additionally, I'd like to confirm that you've tried the following troubleshooting steps:
- Clearing the cache
- Reinstalling the application
- Ensuring all updates are installed

Please let me know if there's anything else you've tried or if you need further assistance. We'll do our best to help you resolve this issue as soon as pos

## 11. Try your own ticket text

In [11]:
my_ticket = "My VPN keeps disconnecting every few minutes since the last update."

hits = retrieve(my_ticket)
answer, secs = ask_ollama(build_prompt(my_ticket, hits))
print(answer)
print(f"\n({secs:.1f}s)")

# See which past tickets were retrieved:
for h in hits:
    print(f"\n[{h['n']}] score={h['score']:.2f} | {h['queue']} / {h['type']} / {h['priority']}")
    print("   ", h['subject'])

Based on the past tickets provided, I would triage the new ticket as follows:

* Queue: Technical Support
* Type: Problem
* Priority: Medium (due to the frequency of disconnections, which suggests a more urgent issue)

Draft First Response:
"Hello <name>, thank you for reaching out about your VPN disconnecting every few minutes since the last update. Unfortunately, I don't have any direct information from our past tickets that would allow me to provide a specific solution. However, I can suggest some troubleshooting steps: please try restarting the LINE and VPN-Router, and also verify that the VPN connection is stable before continuing with your work. Additionally, could you please confirm if there were any recent updates to the router firmware or LINE software? This information will help us better understand the issue and provide a more accurate solution."

[Note: There isn't enough information from past tickets [1], [2], and [3] to directly address this specific issue, as they focuse